In [1]:
!pip install -U openai

In [2]:
from google.colab import userdata
import os
import openai

# Colab에 저장한 Secret에서 API 키 가져와 환경 변수에 등록
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

client = openai.OpenAI()

In [4]:
# 실험에 사용할 역할별 시스템 프롬프트 정의
role_system_prompts = {
    "튜터봇": "당신은 글쓰기 교육 전문가입니다. 학생이 자기소개서를 잘 쓰도록 친절하게 지도해주세요.",
    "상담사": "당신은 공감적인 커리어 상담사입니다. 자기소개서 작성에 부담을 느끼는 이에게 위로와 조언을 해주세요.",
    "면접관": "당신은 기업 인사담당 면접관입니다. 지원자의 자기소개서를 평가하고 개선점을 알려주세요.",
    "비서봇": "당신은 유능한 비서입니다. 자기소개서를 잘 쓰기 위한 요점을 간결하게 정리해서 알려주세요."
}

# 공통 사용자 질문 설정
user_question = "자기소개서를 잘 쓰는 법을 알려줘"

In [5]:
import pandas as pd
import time

results = []  # 결과를 담을 리스트

# 각 역할에 대해 n번씩 응답 생성
n = 3  # 반복 호출 횟수
for role_name, sys_prompt in role_system_prompts.items():
    for i in range(1, n+1):
        messages = [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": user_question}
        ]
        # GPT-4o-mini 모델로 ChatCompletion 생성
        response = client.chat.completions.create(
            messages=messages,
            model="gpt-4o-mini"
        )
        answer = response.choices[0].message.content  # 응답 메시지 추출
        results.append({
            "role": role_name,
            "attempt": i,
            "answer": answer.strip()
        })
        time.sleep(1)  # 1초 간격으로 요청

# 결과를 데이터프레임으로 정리
df = pd.DataFrame(results)
df.head(8)  # 일부 미리보기

,role,attempt,answer
0,튜터봇,1,"자기소개서는 자신을 잘 나타내고, 독자가 여러분에게 관심을 가질 수 있도록 하는 중..."
1,튜터봇,2,"자기소개서는 자신을 효과적으로 표현하고, 다른 사람에게 나를 이해시키는 중요한 글입..."
2,튜터봇,3,자기소개서를 잘 쓰는 것은 자신의 이야기를 효과적으로 전달하는 중요한 과정입니다. ...
3,상담사,1,자기소개서를 쓰는 것이 부담스럽다는 마음 충분히 이해해요. 많은 사람들이 자기소개서...
4,상담사,2,안녕하세요! 자기소개서 작성이 부담스러우시다는 말씀에 공감을 합니다. 많은 분들이 ...
5,상담사,3,"자기소개서를 쓰는 것은 정말 부담스러울 수 있는 일이죠. 특히, 자신을 어떻게 표현..."
6,면접관,1,자기소개서를 잘 쓰기 위해서는 다음과 같은 요소들을 고려해야 합니다:\n\n1. *...
7,면접관,2,자기소개서는 자신의 경험과 능력을 효과적으로 전달하는 중요한 문서입니다. 다음은 자...


In [7]:
# 예시: 각 역할 첫 번째 답변에 대해 임의 평가점수를 부여 (정확성, 공감성, 실용성 각 5점 만점)
evaluation = {
    "튜터봇": {"정확성": 5, "공감성": 3, "실용성": 5},
    "상담사": {"정확성": 4, "공감성": 5, "실용성": 4},
    "면접관": {"정확성": 5, "공감성": 2, "실용성": 5},
    "비서봇": {"정확성": 4, "공감성": 2, "실용성": 5}
}
# DataFrame에서 첫 번째 시도(answer)에 해당하는 행만 추려 평가 테이블 생성
eval_df = pd.DataFrame([
    {"role": role, **scores} for role, scores in evaluation.items()
])
eval_df

,role,정확성,공감성,실용성
0,튜터봇,5,3,5
1,상담사,4,5,4
2,면접관,5,2,5
3,비서봇,4,2,5


In [8]:
results

[{'role': '튜터봇',
  'attempt': 1,
  'answer': '자기소개서는 자신을 잘 나타내고, 독자가 여러분에게 관심을 가질 수 있도록 하는 중요한 문서입니다. 잘 쓰기 위한 몇 가지 팁을 드릴게요.\n\n### 1. **목적 이해하기**\n   - 자기소개서의 목적이 무엇인지 생각해보세요. 학교, 직장, 프로그램 등 지원하는 곳의 특성에 맞게 작성해야 합니다.\n\n### 2. **구성하기**\n   - **서론**: 간단한 인사와 함께 자신을 소개합니다. 왜 이 자기소개서를 쓰게 되었는지 언급해보세요.\n   - **본론**: 자신의 경험, 성격, 가치관 등을 중심으로 이야기합니다.\n     - **경험**: 학업, 봉사활동, 대외활동 등 구체적인 사례를 들어주세요.\n     - **성격**: 자신의 강점과 약점을 솔직하게 이야기하십시오. 약점은 개선하려는 노력도 함께 언급하세요.\n     - **가치관**: 무엇을 중요하게 생각하는지를 강조하세요.\n   - **결론**: 미래에 대한 포부나 목표를 간략하게 정리해 마무리합니다.\n\n### 3. **자신의 이야기를 담기**\n   - 당신의 이야기를 진솔하게 써보세요. 남들과 다른 특별한 경험이나 사건이 있다면 그것을 중심으로 풀어가면 좋습니다.\n\n### 4. **개인적이고 구체적인 사례 사용하기**\n   - 경험을 이야기할 때는 구체적인 상황을 제시해 주시면 좋습니다. 예를 들어, 학교생활이나 사회봉사 등에서의 경험을 바탕으로 작성해보세요.\n\n### 5. **감정과 열정 표현하기**\n   - 자신이 하고 싶은 일이나 목표에 대한 열정을 표현하세요. 감정이 느껴지면 독자가 더 공감하게 됩니다.\n\n### 6. **문법과 표현 체크하기**\n   - 오탈자나 문법 오류가 없는지 확인하세요. 여러 번 읽고, 주변 사람에게 피드백을 받는 것도 좋습니다.\n\n### 7. **형식 준수하기**\n   - 지정된 형식을 따르는 것도 중요합니다. 글자 수, 포맷 등에 유의하면

In [9]:
prompt = f"""
역할에 따른 응답의 일관성과 특성을 다음 평가항목에 대해 예시와 같이 평가(1~5, 5점 만점)를 JSON형식으로 만듭니다.

평가항목:
- 정확성: 각 답변에 제시된 조언이나 정보가 실제 자기소개서 작성에 도움이 되는 타당하고 정확한 내용인가? (예: 튜터봇과 비서봇의 조언은 실제 팁 위주로 정확도가 높을 가능성이 큽니다.)
- 공감성: 답변이 사용자의 감정이나 상황에 공감하고 있나요? (예: 상담사 역할의 답변은 이 측면에서 점수가 높고, 비서봇은 낮을 것입니다.)
- 실용성: 제시된 조언이 구체적이고 실행가능한지 여부를 봅니다. (예: 비서봇과 튜터봇은 실용적 팁을 구조화해서 줄 것이고, 상담사는 다소 추상적인 격려를 포함할 수 있습니다.)

예시:
{{
    "튜터봇": {{"정확성": 5, "공감성": 3, "실용성": 5}},
    "상담사": {{"정확성": 4, "공감성": 5, "실용성": 4}},
    "면접관": {{"정확성": 5, "공감성": 2, "실용성": 5}},
    "비서봇": {{"정확성": 4, "공감성": 2, "실용성": 5}}
}},
    ...

역할(role)에 따른 응답(answer):
{results}
"""

print(prompt)


역할에 따른 응답의 일관성과 특성을 다음 평가항목에 대해 예시와 같이 평가(1~5, 5점 만점)를 JSON형식으로 만듭니다.

평가항목:
- 정확성: 각 답변에 제시된 조언이나 정보가 실제 자기소개서 작성에 도움이 되는 타당하고 정확한 내용인가? (예: 튜터봇과 비서봇의 조언은 실제 팁 위주로 정확도가 높을 가능성이 큽니다.)
- 공감성: 답변이 사용자의 감정이나 상황에 공감하고 있나요? (예: 상담사 역할의 답변은 이 측면에서 점수가 높고, 비서봇은 낮을 것입니다.)
- 실용성: 제시된 조언이 구체적이고 실행가능한지 여부를 봅니다. (예: 비서봇과 튜터봇은 실용적 팁을 구조화해서 줄 것이고, 상담사는 다소 추상적인 격려를 포함할 수 있습니다.)

예시:
{
    "튜터봇": {"정확성": 5, "공감성": 3, "실용성": 5},
    "상담사": {"정확성": 4, "공감성": 5, "실용성": 4},
    "면접관": {"정확성": 5, "공감성": 2, "실용성": 5},
    "비서봇": {"정확성": 4, "공감성": 2, "실용성": 5}
},
    ...

역할(role)에 따른 응답(answer):
[{'role': '튜터봇', 'attempt': 1, 'answer': '자기소개서는 자신을 잘 나타내고, 독자가 여러분에게 관심을 가질 수 있도록 하는 중요한 문서입니다. 잘 쓰기 위한 몇 가지 팁을 드릴게요.\n\n### 1. **목적 이해하기**\n   - 자기소개서의 목적이 무엇인지 생각해보세요. 학교, 직장, 프로그램 등 지원하는 곳의 특성에 맞게 작성해야 합니다.\n\n### 2. **구성하기**\n   - **서론**: 간단한 인사와 함께 자신을 소개합니다. 왜 이 자기소개서를 쓰게 되었는지 언급해보세요.\n   - **본론**: 자신의 경험, 성격, 가치관 등을 중심으로 이야기합니다.\n     - **경험**: 학업, 봉사활동, 대외활동 등 구체적인 사례를 들어주세요.\n     - **성격**: 자신의 강점과 약점을 솔

In [10]:
response = client.chat.completions.create(
    messages=[
        {"role": "user", "content": prompt}
    ],
    model="gpt-4o-mini",
    response_format={"type": "json_object"}
)
evaluation = response.choices[0].message.content
print(evaluation)
print(type(evaluation))

{
    "튜터봇": {
        "정확성": 5,
        "공감성": 3,
        "실용성": 5
    },
    "상담사": {
        "정확성": 4,
        "공감성": 5,
        "실용성": 4
    },
    "면접관": {
        "정확성": 5,
        "공감성": 2,
        "실용성": 5
    },
    "비서봇": {
        "정확성": 4,
        "공감성": 2,
        "실용성": 5
    }
}
<class 'str'>


In [11]:
import json

evals = json.loads(evaluation) # 문자열을 Dictionary로 변환

all_data = []
# Iterate directly through the items in the evals dictionary
for role, scores in evals.items():
    all_data.append({'role': role, **scores})

eval_df = pd.DataFrame(all_data)
eval_df

,role,정확성,공감성,실용성
0,튜터봇,5,3,5
1,상담사,4,5,4
2,면접관,5,2,5
3,비서봇,4,2,5


In [12]:
print(all_data)

[{'role': '튜터봇', '정확성': 5, '공감성': 3, '실용성': 5}, {'role': '상담사', '정확성': 4, '공감성': 5, '실용성': 4}, {'role': '면접관', '정확성': 5, '공감성': 2, '실용성': 5}, {'role': '비서봇', '정확성': 4, '공감성': 2, '실용성': 5}]


In [13]:
import json
# 모든 결과를 JSON 파일로 저장
with open("role_prompt_responses.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)